# MT-SICS Backend Hardware Validation

Run this notebook connected to a physical Mettler Toledo scale to validate
every implemented backend method. Each cell tests one method and logs the
raw MT-SICS command/response.

**Requirements:**
- Physical Mettler Toledo scale connected via USB-to-serial
- Scale powered on and warmed up (60 min)
- Weighing pan empty at start

---
## 1. Setup and Logging

In [1]:
import logging
from datetime import datetime

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO
from pylabrobot.scales import Scale
from pylabrobot.scales.mettler_toledo import MettlerToledoWXS205SDUBackend

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
log_file = f"./logs/hardware_validation/{timestamp}_validation.log"

pylabrobot.verbose(True, level=LOG_LEVEL_IO)
pylabrobot.setup_logger(log_dir=log_file, level=LOG_LEVEL_IO)

plr_logger = logging.getLogger("pylabrobot")
plr_logger.info("=== Hardware validation started ===")

2026-03-30 12:07:26,881 - pylabrobot - INFO - === Hardware validation started ===


In [2]:
# Update port for your system
backend = MettlerToledoWXS205SDUBackend(
  port="/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0"
)
scale = Scale(name="validation_scale", backend=backend, size_x=0, size_y=0, size_z=0)

await scale.setup()

2026-03-30 12:07:26,905 - pylabrobot.io.serial - INFO - Using explicitly provided port: /dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0 (for VID=1027, PID=24577)
2026-03-30 12:07:26,921 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] reset_input_buffer
2026-03-30 12:07:26,923 - pylabrobot - IO - [MT Scale] Sent command: @
2026-03-30 12:07:26,927 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'@\r\n'
2026-03-30 12:07:26,992 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I4 A "B207696838"\r\n'
2026-03-30 12:07:26,993 - pylabrobot - IO - [MT Scale] Received response: b'I4 A "B207696838"\r\n'
2026-03-30 12:07:26,994 - pylabrobot - IO - [MT Scale] Sent command: I0
2026-03-30 12:07:26,995 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I0\r\n'
2026-03-30 12:07:27,025 - pylabrobot.io.serial - IO - [/dev/serial

### Verify setup populated device identity

In [3]:
print(f"Device type:    {backend.device_type}")
print(f"Serial number:  {backend.serial_number}")
print(f"Capacity:       {backend.capacity} g")
print(
  f"Supported commands ({len(backend._supported_commands)}): {sorted(backend._supported_commands)}"
)

assert backend.device_type is not None
assert backend.serial_number is not None
assert backend.capacity > 0
assert "S" in backend._supported_commands
print("PASS: setup populated all identity fields")

Device type:    WXS205SDU WXA-Bridge
Serial number:  B207696838
Capacity:       220.009 g
Supported commands (62): ['@', 'C0', 'C1', 'C2', 'C3', 'COM', 'DAT', 'FCUT', 'FSET', 'I0', 'I1', 'I10', 'I11', 'I14', 'I15', 'I16', 'I2', 'I21', 'I22', 'I23', 'I24', 'I25', 'I26', 'I3', 'I4', 'I5', 'LST', 'M01', 'M02', 'M03', 'M17', 'M18', 'M19', 'M20', 'M21', 'M27', 'M28', 'M29', 'M31', 'M32', 'M33', 'M35', 'RDB', 'S', 'SI', 'SIR', 'SIS', 'SNR', 'SR', 'T', 'TA', 'TAC', 'TI', 'TIM', 'TST0', 'TST1', 'TST2', 'TST3', 'UPD', 'USTB', 'Z', 'ZI']
PASS: setup populated all identity fields


### Supported Python methods on this device

In [4]:
methods = backend.request_supported_methods()
print(f"{len(methods)} methods available on this device:")
for m in methods:
  print(f"  {m}")

36 methods available on this device:
  cancel
  cancel_all
  clear_tare
  deserialize
  get_all_instances
  get_dynamic_weight
  get_serial_number
  get_stable_weight
  get_tare_weight
  get_weight
  get_weight_value_immediately
  measure_temperature
  read_dynamic_weight
  read_stable_weight
  read_weight
  read_weight_value_immediately
  request_capacity
  request_device_type
  request_serial_number
  request_supported_methods
  request_tare_weight
  send_command
  serialize
  set_display_text
  set_host_unit_grams
  set_weight_display
  setup
  stop
  tare
  tare_immediately
  tare_stable
  tare_timeout
  zero
  zero_immediately
  zero_stable
  zero_timeout


### I0 - Discover all implemented commands

This tells us exactly which MT-SICS commands this specific device supports,
resolving whether SC, C, D are truly unsupported or just miscategorized by level.

In [5]:
from collections import defaultdict

# I0 is a multi-response command (B status for each command, A for the last)
responses = await backend.send_command("I0")

print(f"Total commands implemented: {len(responses)}")
print()

# Group by level
by_level = defaultdict(list)
for resp in responses:
  # Format: I0 B/A <level> <"command">
  if len(resp.data) >= 2:
    level = resp.data[0]
    cmd_name = resp.data[1]
    by_level[level].append(cmd_name)

for level in sorted(by_level.keys()):
  cmds = sorted(by_level[level])
  print(f"Level {level} ({len(cmds)} commands): {', '.join(cmds)}")

# Check which commands we use that might not be supported
our_commands = [
  "@",
  "C",
  "D",
  "DW",
  "I0",
  "I1",
  "I2",
  "I4",
  "I50",
  "M21",
  "M28",
  "S",
  "SC",
  "SI",
  "T",
  "TA",
  "TAC",
  "TI",
  "Z",
  "ZC",
  "ZI",
  "TC",
]
all_device_cmds = set()
for cmds in by_level.values():
  all_device_cmds.update(cmds)

print()
print("Commands we use vs device support:")
for cmd in sorted(our_commands):
  status = "supported" if cmd in all_device_cmds else "NOT SUPPORTED"
  print(f"  {cmd}: {status}")

2026-03-30 12:07:28,202 - pylabrobot - IO - [MT Scale] Sent command: I0
2026-03-30 12:07:28,210 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I0\r\n'
2026-03-30 12:07:28,239 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I0 B 0 "I0"\r\n'
2026-03-30 12:07:28,242 - pylabrobot - IO - [MT Scale] Received response: b'I0 B 0 "I0"\r\n'
2026-03-30 12:07:28,255 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I0 B 0 "I1"\r\n'
2026-03-30 12:07:28,256 - pylabrobot - IO - [MT Scale] Received response: b'I0 B 0 "I1"\r\n'
2026-03-30 12:07:28,270 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I0 B 0 "I2"\r\n'
2026-03-30 12:07:28,271 - pylabrobot - IO - [MT Scale] Received response: b'I0 B 0 "I2"\r\n'
2026-03-30 12:07:28,287 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0]

Total commands implemented: 62

Level 0 (12 commands): @, I0, I1, I2, I3, I4, I5, S, SI, SIR, Z, ZI
Level 1 (5 commands): SR, T, TA, TAC, TI
Level 2 (40 commands): C0, C1, C2, C3, COM, DAT, I10, I11, I14, I15, I16, I21, I22, I23, I24, I25, I26, M01, M02, M03, M17, M18, M19, M20, M21, M27, M28, M29, M31, M32, M33, M35, SIS, SNR, TIM, TST0, TST1, TST2, TST3, UPD
Level 3 (5 commands): FCUT, FSET, LST, RDB, USTB

Commands we use vs device support:
  @: supported
  C: NOT SUPPORTED
  D: NOT SUPPORTED
  DW: NOT SUPPORTED
  I0: supported
  I1: supported
  I2: supported
  I4: supported
  I50: NOT SUPPORTED
  M21: supported
  M28: supported
  S: supported
  SC: NOT SUPPORTED
  SI: supported
  T: supported
  TA: supported
  TAC: supported
  TC: NOT SUPPORTED
  TI: supported
  Z: supported
  ZC: NOT SUPPORTED
  ZI: supported


---
## 2. Level 0 Commands (Basic Set)

### @ - Cancel (reset to determined state)

In [6]:
serial_number = await backend.cancel()
print(f"Cancel returned serial number: {serial_number}")
assert serial_number == backend.serial_number
print("PASS: cancel returns correct serial number")

2026-03-30 12:07:29,155 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] reset_input_buffer
2026-03-30 12:07:29,157 - pylabrobot - IO - [MT Scale] Sent command: @
2026-03-30 12:07:29,160 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'@\r\n'
2026-03-30 12:07:29,214 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I4 A "B207696838"\r\n'
2026-03-30 12:07:29,215 - pylabrobot - IO - [MT Scale] Received response: b'I4 A "B207696838"\r\n'


Cancel returned serial number: B207696838
PASS: cancel returns correct serial number


### I4 - Serial number

In [7]:
sn = await backend.request_serial_number()
print(f"Serial number: {sn}")
assert len(sn) > 0
print("PASS: serial number is non-empty")

2026-03-30 12:07:29,220 - pylabrobot - IO - [MT Scale] Sent command: I4
2026-03-30 12:07:29,222 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I4\r\n'
2026-03-30 12:07:29,263 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I4 A "B207696838"\r\n'
2026-03-30 12:07:29,265 - pylabrobot - IO - [MT Scale] Received response: b'I4 A "B207696838"\r\n'


Serial number: B207696838
PASS: serial number is non-empty


### I2 - Device type and capacity

In [8]:
device_type = await backend.request_device_type()
capacity = await backend.request_capacity()
print(f"Device type: {device_type}")
print(f"Capacity: {capacity} g")
assert len(device_type) > 0
assert capacity > 0
print("PASS: device type and capacity valid")

2026-03-30 12:07:29,286 - pylabrobot - IO - [MT Scale] Sent command: I2
2026-03-30 12:07:29,292 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I2\r\n'
2026-03-30 12:07:29,360 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I2 A "WXS205SDU WXA-Bridge 220.00900 g"\r\n'
2026-03-30 12:07:29,361 - pylabrobot - IO - [MT Scale] Received response: b'I2 A "WXS205SDU WXA-Bridge 220.00900 g"\r\n'
2026-03-30 12:07:29,363 - pylabrobot - IO - [MT Scale] Sent command: I2
2026-03-30 12:07:29,364 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I2\r\n'
2026-03-30 12:07:29,422 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I2 A "WXS205SDU WXA-Bridge 220.00900 g"\r\n'
2026-03-30 12:07:29,423 - pylabrobot - IO - [MT Scale] Received response: b'I2 A "WXS205SDU WXA-Bridge 220.00900 g"\r\n'


Device type: WXS205SDU WXA-Bridge
Capacity: 220.009 g
PASS: device type and capacity valid


### I1 - MT-SICS levels

In [9]:
# I1 reports standardized level sets (may not reflect all available commands)
# I0 is the definitive source (already queried during setup)
responses = await backend.send_command("I1")
print(f"I1 reported levels: {responses[0].data}")
print(f"I0 discovered {len(backend._supported_commands)} commands")
print("Note: I1 may underreport - I0 is the authoritative source")

2026-03-30 12:07:29,431 - pylabrobot - IO - [MT Scale] Sent command: I1
2026-03-30 12:07:29,434 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I1\r\n'
2026-03-30 12:07:29,486 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I1 A "01" "2.30" "2.22" "" ""\r\n'
2026-03-30 12:07:29,487 - pylabrobot - IO - [MT Scale] Received response: b'I1 A "01" "2.30" "2.22" "" ""\r\n'


I1 reported levels: ['01', '2.30', '2.22', '', '']
I0 discovered 62 commands
Note: I1 may underreport - I0 is the authoritative source


### S - Stable weight (ensure pan is empty)

In [10]:
weight = await backend.read_stable_weight()
print(f"Stable weight (empty pan): {weight} g")
assert isinstance(weight, float)
print("PASS: stable weight returns float")

2026-03-30 12:07:29,495 - pylabrobot - IO - [MT Scale] Sent command: S
2026-03-30 12:07:29,499 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'S\r\n'
2026-03-30 12:07:29,726 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S    0.00009 g\r\n'
2026-03-30 12:07:29,728 - pylabrobot - IO - [MT Scale] Received response: b'S S    0.00009 g\r\n'


Stable weight (empty pan): 9e-05 g
PASS: stable weight returns float


### SI - Weight immediately

In [11]:
weight = await backend.read_weight_value_immediately()
print(f"Immediate weight: {weight} g")
assert isinstance(weight, float)
print("PASS: immediate weight returns float")

2026-03-30 12:07:29,746 - pylabrobot - IO - [MT Scale] Sent command: SI
2026-03-30 12:07:29,751 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'SI\r\n'
2026-03-30 12:07:29,790 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S    0.00009 g\r\n'
2026-03-30 12:07:29,791 - pylabrobot - IO - [MT Scale] Received response: b'S S    0.00009 g\r\n'


Immediate weight: 9e-05 g
PASS: immediate weight returns float


### Z - Zero (stable)

In [12]:
await scale.zero(timeout="stable")
weight = await backend.read_weight_value_immediately()
print(f"Weight after zero: {weight} g")
assert abs(weight) < 0.001, f"Expected ~0 after zero, got {weight}"
print("PASS: zero sets weight to ~0")

2026-03-30 12:07:29,802 - pylabrobot - IO - [MT Scale] Sent command: Z
2026-03-30 12:07:29,806 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'Z\r\n'
2026-03-30 12:07:30,109 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'Z A\r\n'
2026-03-30 12:07:30,110 - pylabrobot - IO - [MT Scale] Received response: b'Z A\r\n'
2026-03-30 12:07:30,111 - pylabrobot - IO - [MT Scale] Sent command: SI
2026-03-30 12:07:30,112 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'SI\r\n'
2026-03-30 12:07:30,141 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S    0.00000 g\r\n'
2026-03-30 12:07:30,142 - pylabrobot - IO - [MT Scale] Received response: b'S S    0.00000 g\r\n'


Weight after zero: 0.0 g
PASS: zero sets weight to ~0


### ZI - Zero immediately

In [13]:
await scale.zero(timeout=0)
weight = await backend.read_weight_value_immediately()
print(f"Weight after zero immediately: {weight} g")
assert abs(weight) < 0.01
print("PASS: zero immediately sets weight to ~0")

2026-03-30 12:07:30,149 - pylabrobot - IO - [MT Scale] Sent command: ZI
2026-03-30 12:07:30,151 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'ZI\r\n'
2026-03-30 12:07:30,173 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'ZI S\r\n'
2026-03-30 12:07:30,175 - pylabrobot - IO - [MT Scale] Received response: b'ZI S\r\n'
2026-03-30 12:07:30,176 - pylabrobot - IO - [MT Scale] Sent command: SI
2026-03-30 12:07:30,178 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'SI\r\n'
2026-03-30 12:07:30,206 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S    0.00000 g\r\n'
2026-03-30 12:07:30,207 - pylabrobot - IO - [MT Scale] Received response: b'S S    0.00000 g\r\n'


Weight after zero immediately: 0.0 g
PASS: zero immediately sets weight to ~0


---
## 3. Level 1 Commands (Elementary)

**Place a known weight on the scale for tare tests.**

### T - Tare (stable)

In [14]:
input("Place a container on the scale and press Enter...")
await scale.zero(timeout="stable")
await scale.tare(timeout="stable")
tare = await scale.request_tare_weight()
print(f"Tare weight: {tare} g")
assert tare > 0, f"Expected positive tare, got {tare}"
print("PASS: tare stores container weight")

Place a container on the scale and press Enter... 


2026-03-30 12:08:02,193 - pylabrobot - IO - [MT Scale] Sent command: Z
2026-03-30 12:08:02,196 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'Z\r\n'
2026-03-30 12:08:02,545 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'Z A\r\n'
2026-03-30 12:08:02,545 - pylabrobot - IO - [MT Scale] Received response: b'Z A\r\n'
2026-03-30 12:08:02,546 - pylabrobot - IO - [MT Scale] Sent command: T
2026-03-30 12:08:02,548 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'T\r\n'
2026-03-30 12:08:02,960 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'T S    0.00000 g\r\n'
2026-03-30 12:08:02,961 - pylabrobot - IO - [MT Scale] Received response: b'T S    0.00000 g\r\n'
2026-03-30 12:08:02,962 - pylabrobot - IO - [MT Scale] Sent command: TA
2026-03-30 12:08:02,965 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTD

Tare weight: 0.0 g


AssertionError: Expected positive tare, got 0.0

### TA - Request tare weight

In [15]:
tare = await scale.request_tare_weight()
print(f"Tare weight from memory: {tare} g")
assert isinstance(tare, float)
print("PASS: tare weight readable from memory")

2026-03-30 12:08:06,275 - pylabrobot - IO - [MT Scale] Sent command: TA
2026-03-30 12:08:06,277 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'TA\r\n'
2026-03-30 12:08:06,397 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'TA A    0.00000 g\r\n'
2026-03-30 12:08:06,398 - pylabrobot - IO - [MT Scale] Received response: b'TA A    0.00000 g\r\n'


Tare weight from memory: 0.0 g
PASS: tare weight readable from memory


### TAC - Clear tare

In [16]:
await backend.clear_tare()
tare_after_clear = await scale.request_tare_weight()
print(f"Tare after clear: {tare_after_clear} g")
assert abs(tare_after_clear) < 0.001, f"Expected ~0 after clear, got {tare_after_clear}"
print("PASS: clear tare resets tare to 0")

2026-03-30 12:08:07,124 - pylabrobot - IO - [MT Scale] Sent command: TAC
2026-03-30 12:08:07,130 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'TAC\r\n'
2026-03-30 12:08:07,148 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'TAC A\r\n'
2026-03-30 12:08:07,149 - pylabrobot - IO - [MT Scale] Received response: b'TAC A\r\n'
2026-03-30 12:08:07,150 - pylabrobot - IO - [MT Scale] Sent command: TA
2026-03-30 12:08:07,150 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'TA\r\n'
2026-03-30 12:08:07,276 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'TA A    0.00000 g\r\n'
2026-03-30 12:08:07,278 - pylabrobot - IO - [MT Scale] Received response: b'TA A    0.00000 g\r\n'


Tare after clear: 0.0 g
PASS: clear tare resets tare to 0


### SC - Dynamic weight (timed read)

In [17]:
weight = await backend.read_dynamic_weight(timeout=3.0)
print(f"Dynamic weight (3s timeout): {weight} g")
assert isinstance(weight, float)
print("PASS: dynamic weight returns float")

2026-03-30 12:08:07,873 - pylabrobot - IO - [MT Scale] Sent command: SC 3000
2026-03-30 12:08:07,875 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'SC 3000\r\n'
2026-03-30 12:08:07,900 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'ES\r\n'
2026-03-30 12:08:07,901 - pylabrobot - IO - [MT Scale] Received response: b'ES\r\n'


MettlerToledoError: Syntax error: The weigh module/balance has not recognized the received command or the command is not allowed

### C - Cancel all

In [18]:
await backend.cancel_all()
print("PASS: cancel_all completed without error")

2026-03-30 12:08:08,895 - pylabrobot - IO - [MT Scale] Sent command: C
2026-03-30 12:08:08,897 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'C\r\n'
2026-03-30 12:08:08,924 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'ES\r\n'
2026-03-30 12:08:08,924 - pylabrobot - IO - [MT Scale] Received response: b'ES\r\n'


MettlerToledoError: Syntax error: The weigh module/balance has not recognized the received command or the command is not allowed

### D / DW - Display text

In [19]:
import asyncio

await backend.set_display_text("PLR TEST")
await asyncio.sleep(2)
await backend.set_weight_display()
print("PASS: display text set and restored (check terminal if connected)")

2026-03-30 12:08:09,775 - pylabrobot - IO - [MT Scale] Sent command: D "PLR TEST"
2026-03-30 12:08:09,778 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'D "PLR TEST"\r\n'
2026-03-30 12:08:09,803 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'ES\r\n'
2026-03-30 12:08:09,804 - pylabrobot - IO - [MT Scale] Received response: b'ES\r\n'


MettlerToledoError: Syntax error: The weigh module/balance has not recognized the received command or the command is not allowed

### ZC / TC - Timed zero and tare

These are known to return ES (syntax error) on the WXS205SDU despite being in the spec.

In [20]:
from pylabrobot.scales.mettler_toledo.errors import MettlerToledoError

for cmd_name, cmd in [("ZC", "ZC 5000"), ("TC", "TC 5000")]:
  try:
    await backend.send_command(cmd)
    print(f"{cmd_name}: supported on this device")
  except MettlerToledoError as e:
    if "Syntax error" in str(e):
      print(f"{cmd_name}: not supported (ES) - expected for WXS205SDU")
    else:
      print(f"{cmd_name}: unexpected error: {e}")

2026-03-30 12:08:11,168 - pylabrobot - IO - [MT Scale] Sent command: ZC 5000
2026-03-30 12:08:11,172 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'ZC 5000\r\n'
2026-03-30 12:08:11,193 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'ES\r\n'
2026-03-30 12:08:11,194 - pylabrobot - IO - [MT Scale] Received response: b'ES\r\n'
2026-03-30 12:08:11,195 - pylabrobot - IO - [MT Scale] Sent command: TC 5000
2026-03-30 12:08:11,196 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'TC 5000\r\n'
2026-03-30 12:08:11,225 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'ES\r\n'
2026-03-30 12:08:11,227 - pylabrobot - IO - [MT Scale] Received response: b'ES\r\n'


ZC: not supported (ES) - expected for WXS205SDU
TC: not supported (ES) - expected for WXS205SDU


---
## 4. Level 2 Commands (Extended)

### M21 - Set host unit to grams

In [21]:
if "M21" in backend._supported_commands:
  await backend.set_host_unit_grams()
  print("PASS: host unit set to grams")
else:
  print("SKIP: M21 not supported on this device")

2026-03-30 12:08:12,987 - pylabrobot - IO - [MT Scale] Sent command: M21 0 0
2026-03-30 12:08:12,990 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'M21 0 0\r\n'
2026-03-30 12:08:13,016 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M21 A\r\n'
2026-03-30 12:08:13,016 - pylabrobot - IO - [MT Scale] Received response: b'M21 A\r\n'


PASS: host unit set to grams


### I50 - Remaining weighing range (multi-response)

In [22]:
if "I50" in backend._supported_commands:
  remaining = await backend.request_remaining_weighing_range()
  print(f"Remaining weighing range: {remaining} g")
  assert remaining > 0
  assert remaining <= backend.capacity
  print("PASS: remaining range is positive and within capacity")
else:
  print("SKIP: I50 not supported on this device")

SKIP: I50 not supported on this device


### M28 - Temperature sensor

In [23]:
if "M28" in backend._supported_commands:
  temp = await backend.measure_temperature()
  print(f"Scale temperature: {temp} C")
  assert 5 < temp < 50, f"Temperature {temp} C outside reasonable range"
  print("PASS: measure_temperature works - I0 correctly predicted M28 support")
else:
  print("SKIP: M28 not supported on this device")

2026-03-30 12:08:14,914 - pylabrobot - IO - [MT Scale] Sent command: M28
2026-03-30 12:08:14,917 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'M28\r\n'
2026-03-30 12:08:14,950 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M28 B 1 19.9\r\n'
2026-03-30 12:08:14,950 - pylabrobot - IO - [MT Scale] Received response: b'M28 B 1 19.9\r\n'
2026-03-30 12:08:14,966 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M28 A 2 19.4\r\n'
2026-03-30 12:08:14,967 - pylabrobot - IO - [MT Scale] Received response: b'M28 A 2 19.4\r\n'


Scale temperature: 19.9 C
PASS: measure_temperature works - I0 correctly predicted M28 support


---
## 5. Frontend Integration

Test the Scale frontend methods that delegate to the backend.

In [24]:
# Zero via frontend
await scale.zero(timeout="stable")
w = await scale.read_weight(timeout=0)
print(f"Frontend zero + read: {w} g")
assert abs(w) < 0.001

# Tare via frontend
input("Place a container on the scale and press Enter...")
await scale.tare(timeout="stable")
tare = await scale.request_tare_weight()
print(f"Frontend tare weight: {tare} g")
assert tare > 0

# Read via frontend
w = await scale.read_weight(timeout="stable")
print(f"Frontend read (should be ~0 with container tared): {w} g")
assert abs(w) < 0.1

print("PASS: all frontend methods delegate correctly")

2026-03-30 12:08:16,449 - pylabrobot - IO - [MT Scale] Sent command: Z
2026-03-30 12:08:16,453 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'Z\r\n'
2026-03-30 12:08:16,693 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'Z A\r\n'
2026-03-30 12:08:16,694 - pylabrobot - IO - [MT Scale] Received response: b'Z A\r\n'
2026-03-30 12:08:16,694 - pylabrobot - IO - [MT Scale] Sent command: SI
2026-03-30 12:08:16,696 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'SI\r\n'
2026-03-30 12:08:16,741 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S    0.00000 g\r\n'
2026-03-30 12:08:16,749 - pylabrobot - IO - [MT Scale] Received response: b'S S    0.00000 g\r\n'


Frontend zero + read: 0.0 g


Place a container on the scale and press Enter... 


2026-03-30 12:08:17,727 - pylabrobot - IO - [MT Scale] Sent command: T
2026-03-30 12:08:17,729 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'T\r\n'
2026-03-30 12:08:18,099 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'T S    0.00000 g\r\n'
2026-03-30 12:08:18,100 - pylabrobot - IO - [MT Scale] Received response: b'T S    0.00000 g\r\n'
2026-03-30 12:08:18,101 - pylabrobot - IO - [MT Scale] Sent command: TA
2026-03-30 12:08:18,103 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'TA\r\n'
2026-03-30 12:08:18,195 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'TA A    0.00000 g\r\n'
2026-03-30 12:08:18,196 - pylabrobot - IO - [MT Scale] Received response: b'TA A    0.00000 g\r\n'


Frontend tare weight: 0.0 g


AssertionError: 

---
## 6. Performance

In [25]:
import time

import numpy as np

# Stable read latency
times_stable = []
for _ in range(10):
  t0 = time.monotonic_ns()
  await scale.read_weight(timeout="stable")
  t1 = time.monotonic_ns()
  times_stable.append((t1 - t0) / 1e6)

# Immediate read latency
times_immediate = []
for _ in range(10):
  t0 = time.monotonic_ns()
  await scale.read_weight(timeout=0)
  t1 = time.monotonic_ns()
  times_immediate.append((t1 - t0) / 1e6)

print(f"Stable read:    {np.mean(times_stable):.2f} +/- {np.std(times_stable):.2f} ms")
print(f"Immediate read: {np.mean(times_immediate):.2f} +/- {np.std(times_immediate):.2f} ms")

2026-03-30 12:08:20,929 - pylabrobot - IO - [MT Scale] Sent command: S
2026-03-30 12:08:20,930 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'S\r\n'
2026-03-30 12:08:21,040 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S    0.00000 g\r\n'
2026-03-30 12:08:21,041 - pylabrobot - IO - [MT Scale] Received response: b'S S    0.00000 g\r\n'
2026-03-30 12:08:21,042 - pylabrobot - IO - [MT Scale] Sent command: S
2026-03-30 12:08:21,044 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'S\r\n'
2026-03-30 12:08:21,136 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S    0.00000 g\r\n'
2026-03-30 12:08:21,137 - pylabrobot - IO - [MT Scale] Received response: b'S S    0.00000 g\r\n'
2026-03-30 12:08:21,138 - pylabrobot - IO - [MT Scale] Sent command: S
2026-03-30 12:08:21,139 - pylabrobot.io.serial - IO - [

Stable read:    99.29 +/- 6.59 ms
Immediate read: 44.73 +/- 9.15 ms


---
## 7. Teardown

In [26]:
plr_logger.info("=== Hardware validation ended ===")
await scale.stop()
print(f"Log file: {log_file}")

2026-03-30 12:08:23,324 - pylabrobot - INFO - === Hardware validation ended ===


Log file: ./logs/hardware_validation/2026-03-30_12-07-26_validation.log
